### Pacotes importados

### Revisão final

Correção aplicada: o resolvedor por gradiente conjugado do subproblema quadrático foi ajustado para não interromper prematuramente. Agora ele detecta corretamente curvatura não positiva quando a Hessiana não é definida positiva.

In [1]:
using LinearAlgebra
using Printf

## Chapter 10: Newton's local method

### Algorithm 10.1: Newton's local method

**Algorithm 10.1** (Newton's local method):
- **Objective**: Find (an approximation of) a solution to $\\nabla f(x^*) = 0$
- **Input**: gradient $\\nabla f$, Hessian $\\nabla^2 f$, initial $x_0 \\in \\mathbb{R}^n$, precision $\\varepsilon > 0$
- **Initialization**: $k := 0$
- **Repeat**:
  - Calculate $d_k$ solution of $\\nabla^2 f(x_k)\\, d_k = -\\nabla f(x_k)$
  - $x_{k+1} := x_k + d_k$
  - $k := k + 1$
- **Until** $\\|\\nabla f(x_k)\\| \\leq \\varepsilon$

**Example 5.8**: $f(x_1,x_2) = \\frac{1}{2} x_1^2 + x_1 \\cos(x_2)$;  $x_0 = [1.0,\; 1.0]$


In [2]:
# Example 5.8: f(x1, x2) = (1/2)*x1^2 + x1*cos(x2)
# Gradient:  ∇f  = [x1 + cos(x2),  -x1*sin(x2)]
# Hessian:   ∇²f = [[1, -sin(x2)], [-sin(x2), -x1*cos(x2)]]

f(x) = 0.5 * x[1]^2 + x[1] * cos(x[2])

grad_f(x) = [x[1] + cos(x[2]),
             -x[1] * sin(x[2])]

hess_f(x) = [1.0           -sin(x[2]);
             -sin(x[2])    -x[1]*cos(x[2])]

# Algorithm 10.1: Newton's local method
function newton_local(grad, hess, x0; epsilon=1e-6, max_iter=100)
    x = copy(x0)
    k = 0
    println("Algorithm 10.1: Newton's local method")
    println(@sprintf("%-5s %-25s %-20s", "k", "x_k", "||∇f(x_k)||"))
    println("-"^60)

    while true
        g    = grad(x)
        norm_g = norm(g)
        println(@sprintf("%-5d [%12.8f, %12.8f]  %16.8e", k, x[1], x[2], norm_g))

        norm_g <= epsilon && (println("\nConverged after $k iterations"); break)
        k >= max_iter     && (println("\nMax iterations reached");        break)

        H = hess(x)
        d = H \ (-g)       # solve ∇²f(xk)*d = -∇f(xk)
        x = x .+ d
        k += 1
    end

    println("\nx* = $x")
    println("∇f(x*) = $(grad(x))")
    println("∇²f(x*) = $(hess(x))")
    return x
end

# x0 = [1.0, 1.0] as specified in Example 5.8
x0 = [1.0, 1.0]
x_star = newton_local(grad_f, hess_f, x0)


Algorithm 10.1: Newton's local method
k     x_k                       ||∇f(x_k)||         
------------------------------------------------------------
0     [  1.00000000,   1.00000000]    1.75516512e+00
1     [ -0.23384513,   1.36419221]    2.30665382e-01
2     [  0.01081438,   1.58483641]    1.12840545e-02
3     [ -0.00000213,   1.57079327]    2.32349802e-06
4     [  0.00000000,   1.57079633]    8.35430835e-17

Converged after 4 iterations

x* = [1.990485074400243e-17, 1.5707963267948966]
∇f(x*) = [8.113719070137009e-17, -1.990485074400243e-17]
∇²f(x*) = [1.0 -1.0; -1.0 -1.2188205875574195e-33]


2-element Vector{Float64}:
 1.990485074400243e-17
 1.5707963267948966

### Algorithm 10.2: Newton's local method by quadratic modeling

**Algorithm 10.2** minimises the quadratic model at each iterate:

$$m_{x_k}(x_k + d) = f(x_k) + d^\top \nabla f(x_k) + \frac{1}{2}\, d^\top \nabla^2 f(x_k)\, d$$

The minimiser satisfies $d_k = \operatorname{argmin}_d\, m_{x_k}(x_k+d)$, solved via

* **Algorithm 9.1** – direct method (Cholesky factorisation), or  
* **Algorithm 9.2** – conjugate gradient method.

Both require $\nabla^2 f(x_k)$ to be **positive definite**.

We test Algorithm 10.2 on the **Rosenbrock function** ($n$ variables):

$$f(x) = \sum_{i=1}^{n-1}\bigl[100\,(x_{i+1}-x_i^2)^2 + (1-x_i)^2\bigr]$$

Partial derivatives of each term $f_i$:

$$\frac{\partial f_i}{\partial x_i} = -400\,x_i(x_{i+1}-x_i^2) - 2(1-x_i), \qquad
  \frac{\partial f_i}{\partial x_{i+1}} = 200\,(x_{i+1}-x_i^2)$$

$$\frac{\partial^2 f_i}{\partial x_i^2} = -400\,x_{i+1}+1200\,x_i^2+2, \qquad
  \frac{\partial^2 f_i}{\partial x_i\,\partial x_{i+1}} = -400\,x_i, \qquad
  \frac{\partial^2 f_i}{\partial x_{i+1}^2} = 200$$


In [ ]:
# ── Rosenbrock function (n variables) ──────────────────────────────────────
rosenbrock(x) = sum(100*(x[i+1]-x[i]^2)^2 + (1-x[i])^2 for i in 1:length(x)-1)

function grad_rosenbrock(x)
    n = length(x);  g = zeros(n)
    for i in 1:n-1
        g[i]   += -400*x[i]*(x[i+1]-x[i]^2) - 2*(1-x[i])
        g[i+1] += 200*(x[i+1]-x[i]^2)
    end
    return g
end

function hess_rosenbrock(x)
    n = length(x);  H = zeros(n,n)
    for i in 1:n-1
        H[i,i]     += -400*x[i+1] + 1200*x[i]^2 + 2
        H[i,i+1]   += -400*x[i]
        H[i+1,i]   += -400*x[i]
        H[i+1,i+1] += 200
    end
    return H
end

# ── Algorithm 9.1: direct method (Cholesky) ────────────────────────────────
# Solves  min_{d}  ½ d'Qd + b'd   ⟺  Q d = -b   (Q must be pos. def.)
function quadratic_direct(Q, b)
    L = cholesky(Q)          # throws PosDefException if not positive definite
    return L \ (-b)
end

# ── Algorithm 9.2: conjugate gradient method ───────────────────────────────
# Solves  min_{d}  ½ d'Qd + b'd   ⟺  Q d = -b   (Q must be pos. def.)
function quadratic_cg(Q, b; tol=1e-12)
    n = length(b)
    x = zeros(n)
    gk = Q*x .+ b
    d = -gk
    k = 1

    while norm(gk) > tol && k <= n + 1
        Qd  = Q*d
        dQd = dot(d, Qd)

        # O CG para o subproblema quadrático exige Q definida positiva.
        # Se d'Qd <= 0, a direção revela curvatura não positiva e o modelo
        # quadrático não possui mínimo finito nessa direção.
        if dQd <= 0
            error("Non-positive curvature detected: d'Qd = $dQd. Hessian is not positive definite.")
        end

        alpha = -dot(d, gk) / dQd
        x = x .+ alpha .* d

        gk1 = Q*x .+ b
        if norm(gk1) <= tol
            return x
        end

        beta = dot(gk1, gk1) / dot(gk, gk)
        d = -gk1 .+ beta .* d
        gk = gk1
        k += 1
    end

    if norm(gk) > tol
        error("Conjugate Gradient did not converge within $(n + 1) iterations.")
    end

    return x
end

# ── Algorithm 10.2 ─────────────────────────────────────────────────────────
function newton_quadratic(grad, hess, x0;
                          epsilon=1e-6, max_iter=200, method=:direct)
    x    = copy(x0);  k = 0
    mname = method == :direct ? "Direct (Cholesky)" : "Conjugate Gradient"
    println("Algorithm 10.2: Newton's local method by quadratic modeling")
    println("Quadratic subproblem solver: $mname")
    println(@sprintf("%-5s %-50s %-20s", "k", "x_k", "||∇f(x_k)||"))
    println("-"^80)

    while true
        g    = grad(x);  ng = norm(g)
        xs   = "[" * join([@sprintf("%10.6f", xi) for xi in x], ", ") * "]"
        println(@sprintf("%-5d %-50s %16.8e", k, xs, ng))

        ng <= epsilon && (println("\nConverged after $k iterations"); break)
        k >= max_iter && (println("\nMax iterations reached");        break)

        H = hess(x)
        d = method == :direct ? quadratic_direct(H, g) : quadratic_cg(H, g)
        x = x .+ d;  k += 1
    end

    println("\nx* = $x")
    println("∇f(x*) = $(grad(x))")
    return x
end

# ── Test on Rosenbrock, n = 2 ───────────────────────────────────────────────
println("=" ^ 80)
println("Rosenbrock function (n=2), x0 = [-1.0, -1.0]")
println("=" ^ 80)
x0_rb  = [-1.0, -1.0]
x_rb   = newton_quadratic(grad_rosenbrock, hess_rosenbrock, x0_rb;
                           epsilon=1e-10)


We now apply the algorithm on example 5.8. In this case, the algorithm fails to converge, and one hessian is not positive definite. We try first using the direct method to solve the quadratic problem. An error is triggered.

In [4]:
# Apply Algorithm 10.2 to Example 5.8 – direct method (Cholesky)
# f(x1,x2) = ½x1² + x1·cos(x2),  x0 = [1.0, 1.0]

println("=" ^ 70)
println("Example 5.8 with Algorithm 10.2 - Direct method (Cholesky)")
println("=" ^ 70)
println("x0 = [1.0, 1.0]\n")

x0_ex = [1.0, 1.0]
H0    = hess_f(x0_ex)
println("Hessian at x0 = [1, 1]:")
println(H0)
println("Eigenvalues: ", eigvals(H0))
println("The Hessian is NOT positive definite (has negative eigenvalue).\n")

try
    newton_quadratic(grad_f, hess_f, x0_ex; method=:direct)
catch e
    println("ERROR caught (as expected): ", typeof(e))
    println("Message: ", e)
    println()
    println("Explanation: Algorithm 10.2 cannot be applied because ∇²f(x0) is not")
    println("positive definite. The quadratic model is unbounded from below.")
    println("The Cholesky factorization fails, triggering a PosDefException.")
end


Example 5.8 with Algorithm 10.2 - Direct method (Cholesky)
x0 = [1.0, 1.0]

Hessian at x0 = [1, 1]:
[1.0 -0.8414709848078965; -0.8414709848078965 -0.5403023058681398]
Eigenvalues: [-0.910855416378039, 1.3705531105098994]
The Hessian is NOT positive definite (has negative eigenvalue).

Algorithm 10.2: Newton's local method by quadratic modeling
Quadratic subproblem solver: Direct (Cholesky)
k     x_k                                                ||∇f(x_k)||         
--------------------------------------------------------------------------------
0     [  1.000000,   1.000000]                             1.75516512e+00
ERROR caught (as expected): PosDefException
Message: PosDefException(2)

Explanation: Algorithm 10.2 cannot be applied because ∇²f(x0) is not
positive definite. The quadratic model is unbounded from below.
The Cholesky factorization fails, triggering a PosDefException.


If we try with the conjugate gradient method, an error is also triggered.

In [ ]:
# Apply Algorithm 10.2 to Example 5.8 – conjugate gradient method

println("=" ^ 70)
println("Example 5.8 with Algorithm 10.2 - Conjugate Gradient method")
println("=" ^ 70)
println("x0 = [1.0, 1.0]\n")

try
    newton_quadratic(grad_f, hess_f, [1.0, 1.0]; method=:cg)
catch e
    println("ERROR caught (as expected): ", typeof(e))
    println("Message: ", e)
    println()
    println("Explanation: The conjugate gradient method also fails because it detects")
    println("a non-positive curvature direction (d'Qd ≤ 0) when the Hessian is not")
    println("positive definite. The quadratic model has no minimum.")
end

println()
println("=" ^ 70)
println("Summary")
println("=" ^ 70)
println("- Algorithm 10.1 works on Example 5.8: converges to a SADDLE POINT.")
println("- Algorithm 10.2 FAILS on Example 5.8: Hessian at x0 is not positive")
println("  definite, so the quadratic model is unbounded from below.")
println("- Algorithm 10.2 works on Rosenbrock: converges to the global minimum [1,1].")
println()
println("This illustrates that Algorithms 10.1 and 10.2 are equivalent ONLY when")
println("the Hessian is positive definite at each iterate.")
